In [7]:
import sys
import os

In [8]:
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/nlp'):
        !git clone https://github.com/jaYulichka46/nlp.git
    
    %cd /content/nlp
    !pip install regex pandas stanza scikit-learn -q
    sys.path.append('/content/nlp')

    FOLDER_ID = '1Xhu4xjZpRu-RP730-hyErp5F0C3l_EvO'
    
    os.makedirs('data', exist_ok=True)
    !gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O data/
    
    data_dir = 'data/sample'
else:
    sys.path.append(os.path.abspath('..'))
    data_dir = '../data'

In [9]:
import pandas as pd
import stanza
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import json
from src.ling_features import get_stanza_pipeline, extract_ling_features


In [12]:
sample_data_path = os.path.join(data_dir, "sample", "sample_raw.csv")

if os.path.exists(sample_data_path):
    df_sample = pd.read_csv(sample_data_path)
    
    text_col = 'clean_text' 
    label_col = 'target'
    
    df_sample = df_sample.dropna(subset=[text_col, label_col]).reset_index(drop=True)

    if len(df_sample) > 1000:
        df_sample = df_sample.sample(n=1000, random_state=42).reset_index(drop=True)
        
    display(df_sample[[text_col, label_col]].head(10))
else:
    print(f" File {sample_data_path} not found")

,clean_text,target
0,"Головний тренер солігорського ""Шахтаря"" Юрій В...",спорт
1,Про це на своїй сторінці у Facebook написав пр...,новини
2,Про це повідомляється в доповіді некомерційної...,новини
3,Легенда НБА Шакіл О'Ніл продав свій маєток у Ф...,спорт
4,Засновник фінансової піраміди B2B Jewelry Мико...,бізнес
5,Про це повідомив голова Донецької ОДА Павло Ки...,новини
6,"Творці стрічки ""Поводир"" дають змогу людям із ...",новини
7,Про це на своїй сторінці у Facebook повідомили...,новини
8,"Звернення оприлюднив народний депутат України,...",політика
9,Американська корпорація Facebook запустила в 3...,технології


In [13]:
nlp = get_stanza_pipeline(use_gpu=True if 'google.colab' in sys.modules else False)

lemmas = []
pos_seqs = []

for text in df_sample[text_col]:
    res = extract_ling_features(str(text), nlp)
    lemmas.append(res['lemma_text'])
    pos_seqs.append(res['pos_seq'])

df_sample['lemma_text'] = lemmas
df_sample['pos_seq'] = pos_seqs

display(df_sample[[text_col, 'lemma_text', 'pos_seq']].head(2))

2026-03-04 02:04:50 INFO: Downloaded file to C:\Users\mfese\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\resources.json
2026-03-04 02:04:50 WARNING: Language uk package default expects mwt, which has been added
2026-03-04 02:04:50 INFO: Downloading these customized packages for language: uk (Ukrainian)...
| Processor       | Package     |
---------------------------------
| tokenize        | iu          |
| mwt             | iu          |
| pos             | iu_charlm   |
| lemma           | iu_nocharlm |
| pretrain        | conll17     |
| backward_charlm | conll17     |
| forward_charlm  | conll17     |

2026-03-04 02:04:50 INFO: File exists: C:\Users\mfese\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\uk\tokenize\iu.pt
2026-03-04 02:04:50 INFO: File exists: C:\Users\mfese\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\uk\mwt\iu.pt
2026-03-04 02:04:50 INFO: File exists: C:\Users\mfese\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\uk\pos\iu_char

,clean_text,lemma_text,pos_seq
0,"Головний тренер солігорського ""Шахтаря"" Юрій В...","головний тренер солігорський "" Шахтаря "" Юрій ...",ADJ NOUN ADJ PUNCT PROPN PUNCT PROPN PROPN ADP...
1,Про це на своїй сторінці у Facebook написав пр...,про це на свій сторінка у Facebook написати пр...,ADP PRON ADP DET NOUN ADP X VERB NOUN NOUN PRO...


In [14]:
def evaluate_baseline(X, y, feature_name):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    vectorizer = TfidfVectorizer(max_features=5000)
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)
    
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train_vec, y_train)
    
    y_pred = clf.predict(X_test_vec)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    vocab_size = len(vectorizer.vocabulary_)
    
    return acc, f1, vocab_size

In [15]:
acc_raw, f1_raw, vocab_raw = evaluate_baseline(df_sample[text_col], df_sample[label_col], "Raw Text")
acc_lemma, f1_lemma, vocab_lemma = evaluate_baseline(df_sample['lemma_text'], df_sample[label_col], "Lemma Text")

print(f"Baseline 1 (Raw): Accuracy = {acc_raw:.4f}, Macro-F1 = {f1_raw:.4f}, Словник: {vocab_raw} слів")
print(f"Baseline 2 (Lemma): Accuracy = {acc_lemma:.4f}, Macro-F1 = {f1_lemma:.4f}, Словник: {vocab_lemma} слів")

Baseline 1 (Raw): Accuracy = 0.4500, Macro-F1 = 0.2842, Словник: 5000 слів
Baseline 2 (Lemma): Accuracy = 0.4500, Macro-F1 = 0.2842, Словник: 4830 слів


In [16]:
edge_cases_path = "tests/ling_edge_cases.jsonl" if 'google.colab' in sys.modules else "../tests/ling_edge_cases.jsonl"

if os.path.exists(edge_cases_path):
    with open(edge_cases_path, "r", encoding="utf-8") as f:
        for line in f:
            case = json.loads(line.strip())
            res = extract_ling_features(case["text"], nlp)
            
            print(f"Text: {case['text']}")
            print(f"Lemma:    {res['lemma_text']}")
            print(f"Pos:     {res['pos_seq']}")
            print(f"Expected issue: {case['expected_issue']}")
            print("-" * 80)
else:
    print(f"File {edge_cases_path} not found")

Text: Бійці ЗСУ та спецпризначенці НАБУ провели спільну операцію.
Lemma:    бійка ЗСУ та спецпризначенець НАБУ провести спільний операція .
Pos:     NOUN PROPN CCONJ NOUN PROPN VERB ADJ NOUN PUNCT
Expected issue: Абревіатури: Stanza може не розпізнати ЗСУ та НАБУ як власні назви (PROPN) і дати хибну лему.
--------------------------------------------------------------------------------
Text: Міський голова Івано-Франківська звернувся до Кабміну.
Lemma:    міський голова івано-франківський звернутися до Кабмін .
Pos:     ADJ NOUN PROPN VERB ADP PROPN PUNCT
Expected issue: Власні назви з дефісом: 'Івано-Франківська' може розбитися на 'Івано', '-', 'Франківськ', руйнуючи цілісність сутності.
--------------------------------------------------------------------------------
Text: Через ракетні обстріли країні загрожує повний блекаут.
Lemma:    через ракетний обстріл країна загрожувати повний блекаут .
Pos:     ADP ADJ NOUN NOUN VERB ADJ NOUN PUNCT
Expected issue: Неологізми/Англіцизми: 'блека

In [17]:
summary_path = "docs/audit_summary_lab3.md" if 'google.colab' in sys.modules else "../docs/audit_summary_lab3.md"

markdown_content = f"""# Audit Summary Lab 3: Lemma & POS Tagging (News Classification)

## 1. Аналіз помилок (Error Analysis)
При обробці українських новин бібліотекою Stanza виявлено специфічні проблеми:
* **Абревіатури та трансліт:** Складні власні назви (ЗСУ, НАБУ) іноді отримують хибні леми.
* **Омонімія:** Слова типу "бійці" лематизуються як "бійка", що спотворює точний граматичний сенс.
* **Числівники та пунктуація:** Дробові числа часто розбиваються пробілами.

## 2. Baseline Порівняння (Класифікація)
Проведено порівняння класифікатора (Logistic Regression + TF-IDF) на тестовій вибірці новин.
* **Baseline 1 (Raw Text):** Accuracy = {acc_raw:.4f}, Macro-F1 = {f1_raw:.4f}, Розмір словника = {vocab_raw}.
* **Baseline 2 (Lemma Text):** Accuracy = {acc_lemma:.4f}, Macro-F1 = {f1_lemma:.4f}, Розмір словника = {vocab_lemma}.

## 3. Висновок: Коли леми додають цінність
Незважаючи на помилки в Edge Cases (особливо з абревіатурами), для задачі класифікації текстів лематизація виявилася **корисною / нейтральною** (залежить від метрик вище).
Леми нормалізують словоформи (наприклад, зводять "міністр", "міністра", "міністру" до одного токена), що дозволяє TF-IDF моделі працювати з більш щільною матрицею ознак і меншим словником.
**Рішення для проєкту:** Для задачі класифікації новин **ми приймаємо лематизацію**. Вона зменшує зашумленість даних, хоча для точного витягування сутностей (NER) ми б від неї відмовилися.
"""

os.makedirs(os.path.dirname(summary_path), exist_ok=True)
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(markdown_content)